In [1]:
import json
import random
from datetime import datetime
from typing import Dict, List, Tuple

INPUT_JSONL = "data/openalex_enriched.jsonl"
OUTPUT_QUERIES = "data/queries.jsonl"
OUTPUT_QRELS = "data/qrels.jsonl"
OUTPUT_META = "data/query_qrels_metadata.json"

RANDOM_SEED = 70
CANDIDATE_QUERY_COUNT = 250
MIN_RELEVANT_CITATIONS = 3


def load_jsonl(filepath: str) -> List[Dict]:
    records = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def write_jsonl(records: List[Dict], filepath: str) -> None:
    with open(filepath, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def filter_query_candidates(records: List[Dict]) -> List[Dict]:
    """
    Keep only records that:
    - matched to OpenAlex
    - have at least MIN_RELEVANT_CITATIONS within the corpus
    """
    candidates = []
    for record in records:
        if record.get("openalex_match_status") != "matched":
            continue
        if record.get("relevant_citation_count_in_corpus", 0) < MIN_RELEVANT_CITATIONS:
            continue
        candidates.append(record)
    return candidates


def build_queries(records: List[Dict], candidate_query_count: int) -> List[Dict]:
    random.seed(RANDOM_SEED)

    candidates = filter_query_candidates(records)

    if len(candidates) < candidate_query_count:
        sampled = candidates
    else:
        sampled = random.sample(candidates, candidate_query_count)

    queries = []
    for idx, record in enumerate(sampled, start=1):
        query_id = f"Q{idx:03d}"
        query_text = f"{record['title']} {record['abstract']}".strip()

        queries.append({
            "query_id": query_id,
            "paper_id": record["paper_id"],
            "query_text": query_text,
            "title": record["title"],
            "abstract": record["abstract"],
            "primary_category": record.get("primary_category"),
            "published_year": record.get("published_year"),
            "relevance_source": "citation"
        })

    return queries


def build_qrels(queries: List[Dict], record_map: Dict[str, Dict]) -> List[Dict]:
    qrels = []

    for query in queries:
        paper_id = query["paper_id"]
        query_id = query["query_id"]

        relevant_docs = record_map[paper_id].get("relevant_citations_in_corpus", [])

        for doc_id in relevant_docs:
            qrels.append({
                "query_id": query_id,
                "doc_id": doc_id,
                "relevance": 1,
                "source": "citation"
            })

    return qrels


def build_metadata(records: List[Dict], queries: List[Dict], qrels: List[Dict]) -> Dict:
    query_relevance_counts = []
    qrel_map = {}

    for qrel in qrels:
        qid = qrel["query_id"]
        qrel_map[qid] = qrel_map.get(qid, 0) + 1

    for query in queries:
        query_relevance_counts.append(qrel_map.get(query["query_id"], 0))

    return {
        "generated_timestamp_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
        "input_records": len(records),
        "candidate_query_target": CANDIDATE_QUERY_COUNT,
        "min_relevant_citations": MIN_RELEVANT_CITATIONS,
        "final_query_count": len(queries),
        "final_qrel_count": len(qrels),
        "average_relevant_docs_per_query": round(
            sum(query_relevance_counts) / len(query_relevance_counts), 2
        ) if query_relevance_counts else 0,
        "random_seed": RANDOM_SEED,
        "relevance_source": "citation"
    }


def main() -> None:
    records = load_jsonl(INPUT_JSONL)
    record_map = {r["paper_id"]: r for r in records}

    queries = build_queries(records, CANDIDATE_QUERY_COUNT)
    qrels = build_qrels(queries, record_map)
    metadata = build_metadata(records, queries, qrels)

    write_jsonl(queries, OUTPUT_QUERIES)
    write_jsonl(qrels, OUTPUT_QRELS)

    with open(OUTPUT_META, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(queries)} queries to {OUTPUT_QUERIES}")
    print(f"Saved {len(qrels)} qrels to {OUTPUT_QRELS}")
    print(f"Saved metadata to {OUTPUT_META}")
    print(json.dumps(metadata, indent=2))


if __name__ == "__main__":
    main()

Saved 250 queries to data/queries.jsonl
Saved 1478 qrels to data/qrels.jsonl
Saved metadata to data/query_qrels_metadata.json
{
  "generated_timestamp_utc": "2026-03-13T22:55:58Z",
  "input_records": 15000,
  "candidate_query_target": 250,
  "min_relevant_citations": 3,
  "final_query_count": 250,
  "final_qrel_count": 1478,
  "average_relevant_docs_per_query": 5.91,
  "random_seed": 70,
  "relevance_source": "citation"
}
